# 09 — Quiz

The quiz layer: how many generated questions survive the round-trip check (`quiz.py`), and how well `quiz.grade()` marks a student's free-text answer against the reference.

**Running it.** Every section below is the evaluation's own code. With `RUN = False`
(the default) nothing is recomputed: the results saved in `data/eval/` are loaded
and shown. Set `RUN = True` in the first code cell to measure again, which
overwrites those files. Quiz generation takes about 3 minutes; the grader calibration seconds.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("") if os.path.basename(os.path.abspath("")) == "notebooks"
                else os.path.join(os.path.abspath(""), "notebooks"))
from eval_common import repo_root  # noqa: E402

REPO_ROOT = repo_root()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

# The application's settings (backend/service.py). They are read when the
# pipeline modules are imported, so they are set before anything else.
os.environ.setdefault("SA_EMBEDDER", "bge-small")

# False: show the results saved in data/eval. True: run the evaluation again
# and overwrite them (the run time is given at the top of the notebook).
RUN = False

In [2]:
import json

import pandas as pd

EVAL_DIR = REPO_ROOT / "data" / "eval"


def saved(name):
    """A results file from data/eval."""
    return json.loads((EVAL_DIR / name).read_text(encoding="utf-8"))

## Quiz question generation

Evaluates quiz question generation (quiz.py) on the project's documents: the
four papers used for the retrieval evaluation plus the six-page sample
lecture notes, indexed together as one subject.

Two modes:

```text
  generate (default)
      For every topic, up to ATTEMPTS_PER_TOPIC chunks are turned into
      question/answer pairs. Every attempt is logged with why it was rejected,
      and well-formed pairs are round-trip checked separately so that pairs
      the filter rejects are kept for comparison. Also records whether
      retrieval, given the generated question, returns the source page.
      Writes data/eval/quiz_generation.json and a blind rating sheet,
      data/eval/quiz_rating_sheet.csv, which does not show the filter result.

  score
      After the rating sheet has been filled in by a person (1 = yes,
      0 = no in each rating column), compares the ratings of pairs the
      round-trip filter kept with those it rejected.
          python backend/scripts/eval_quiz_generation.py score
```

In [3]:
# The script's command-line options, as it would have read them.
MODE = "generate"        # "generate" writes questions; "score" reads back a rated sheet
sys.argv = ["notebook"] + (["score"] if MODE == "score" else [])

In [4]:
import csv
import json
import os
import random
import sys
import time
from collections import Counter
from pathlib import Path
from statistics import mean

ROOT = REPO_ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

EVAL_DIR = ROOT / "data" / "eval"
from backend.pipeline.generator import MODEL_NAME  # noqa: E402
# The default model writes the unsuffixed files; any other model (qwen3:14b
# with SA_BACKEND=ollama, say) gets its own, so a comparison run never
# overwrites a sheet that is being rated.
MODEL_TAG = ("" if MODEL_NAME == "Qwen/Qwen2.5-1.5B-Instruct"
             else "_" + MODEL_NAME.split("/")[-1].lower().replace(":", "-"))
OUT_PATH = EVAL_DIR / f"quiz_generation{MODEL_TAG}.json"
SHEET_PATH = EVAL_DIR / f"quiz_rating_sheet{MODEL_TAG}.csv"
SCORES_PATH = EVAL_DIR / f"quiz_rating_results{MODEL_TAG}.json"

DOCUMENTS = [
    ROOT / "data" / "raw" / "embedding.pdf",
    ROOT / "data" / "raw" / "Whisper.pdf",
    ROOT / "data" / "raw" / "Flant5pdf.pdf",
    ROOT / "data" / "raw" / "Hallucinations_in_Large_Language_Models_LLMs.pdf",
    ROOT / "data" / "Prototype" / "sample_lecture_notes.pdf",
]
ATTEMPTS_PER_TOPIC = 3
CHUNKING = "sentence"    # as backend/service.py
os.environ.setdefault("SA_EMBEDDER", "bge-small")   # as backend/service.py
TOP_K = 3
SEED = 7
RATINGS = {
    "clear": "The question is grammatical and unambiguous",
    "answerable": "The passage contains the answer to the question",
    "answer_correct": "The reference answer is correct for the question",
    "useful": "The question tests something worth revising (not trivia)",
}

In [5]:
def generate():
    import faiss
    import numpy as np

    from backend.pipeline import quiz
    from backend.pipeline.embedder import embed
    from backend.pipeline.generator import complete, generate as answer
    from backend.pipeline.loader import load_file
    from backend.pipeline.preprocessor import preprocess
    from backend.pipeline.retriever import retrieve

    chunks = []
    for path in DOCUMENTS:
        chunks.extend(preprocess(load_file(str(path)), chunking=CHUNKING))
    embeddings = embed(chunks)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.ascontiguousarray(embeddings, dtype=np.float32))

    topics = quiz.build_topics(chunks)
    usable = sum(len(t.chunk_indices) for t in topics)
    print(f"{len(chunks)} chunks, {usable} usable for quizzes, {len(topics)} topics")

    rng = random.Random(SEED)
    attempts = []
    started = time.time()
    for topic in topics:
        order = list(topic.chunk_indices)
        rng.shuffle(order)
        for i in order[:ATTEMPTS_PER_TOPIC]:
            chunk = chunks[i]
            t0 = time.time()
            item, reason = quiz.generate_item(
                chunk, answer_fn=answer,
                question_fn=lambda p: complete(p, max_new_tokens=48))
            record = {"topic_id": topic.id, "chunk_index": i,
                      "source_file": chunk.source_file, "page": chunk.page,
                      "rejected_because": reason}
            if item is not None:
                retrieved = retrieve(item.question, index, chunks, k=TOP_K)
                passed = quiz.roundtrip_check(item, answer, lambda q: retrieved)
                record.update({
                    "question": item.question,
                    "answer": item.answer,
                    "passage": item.passage,
                    "section": item.section,
                    "roundtrip_answer": item.roundtrip_answer,
                    "roundtrip_score": item.roundtrip_score,
                    "passed_roundtrip": passed,
                    "source_page_retrieved": any(
                        (c.source_file, c.page) == (chunk.source_file, chunk.page)
                        for c in retrieved),
                })
                if not passed:
                    record["rejected_because"] = "failed round-trip check"
            record["seconds"] = round(time.time() - t0, 2)
            attempts.append(record)
            print(f"  {record['rejected_because'] or 'kept':<36} "
                  f"{record.get('question', '')[:70]}")
    elapsed = time.time() - started

    formed = [a for a in attempts if "question" in a]
    kept = [a for a in formed if a["passed_roundtrip"]]
    per_doc = {}
    for a in attempts:
        d = per_doc.setdefault(a["source_file"], Counter())
        d["attempts"] += 1
        d["kept"] += a["rejected_because"] is None
    summary = {
        "chunks": len(chunks),
        "usable_chunks": usable,
        "topics": len(topics),
        "attempts": len(attempts),
        "outcomes": dict(Counter(a["rejected_because"] or "kept" for a in attempts)),
        "well_formed_rate": round(len(formed) / len(attempts), 3),
        "roundtrip_pass_rate_of_well_formed": round(len(kept) / len(formed), 3),
        "kept_rate": round(len(kept) / len(attempts), 3),
        "topics_with_at_least_one_item": len({a["topic_id"] for a in kept}),
        "source_page_retrieved": {
            "kept": round(mean(a["source_page_retrieved"] for a in kept), 3),
            "rejected": round(mean(a["source_page_retrieved"] for a in formed
                                   if not a["passed_roundtrip"]), 3),
        },
        "seconds_per_attempt": round(elapsed / len(attempts), 2),
        "per_document": {k: dict(v) for k, v in per_doc.items()},
    }
    print(json.dumps(summary, indent=2))

    OUT_PATH.write_text(json.dumps({
        "note": "Quiz generation over the evaluation papers and sample lecture "
                "notes. Round-trip check applied separately so rejected pairs "
                "are kept for comparison.",
        "settings": {"model": MODEL_NAME, "attempts_per_topic": ATTEMPTS_PER_TOPIC, "top_k": TOP_K,
                     "chunking": CHUNKING,
                     "embedder": os.environ["SA_EMBEDDER"],
                     "seed": SEED, "grade_threshold": quiz.GRADE_THRESHOLD,
                     "device": os.environ.get("SA_DEVICE", "gpu")},
        "summary": summary,
        "attempts": attempts,
    }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

    sheet = [dict(a, id=n) for n, a in enumerate(formed, start=1)]
    rng.shuffle(sheet)
    with open(SHEET_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "document", "section", "passage", "question",
                         "reference_answer", *RATINGS, "notes"])
        for a in sheet:
            writer.writerow([a["id"], a["source_file"], a["section"], a["passage"],
                             a["question"], a["answer"], *[""] * len(RATINGS), ""])
    print(f"\n  -> {OUT_PATH.relative_to(ROOT)}\n  -> {SHEET_PATH.relative_to(ROOT)} "
          f"({len(sheet)} pairs to rate: {', '.join(RATINGS)})")

In [6]:
def score():
    data = json.loads(OUT_PATH.read_text(encoding="utf-8"))
    formed = [a for a in data["attempts"] if "question" in a]
    by_id = {n: a for n, a in enumerate(formed, start=1)}

    rated = []
    with open(SHEET_PATH, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            values = {k: row[k].strip() for k in RATINGS}
            if all(v in ("0", "1") for v in values.values()):
                a = by_id[int(row["id"])]
                rated.append({"kept": a["passed_roundtrip"],
                              **{k: int(v) for k, v in values.items()}})
    if not rated:
        raise SystemExit("No fully rated rows yet — fill in 0/1 for every rating column.")

    groups = {"kept by filter": [r for r in rated if r["kept"]],
              "rejected by filter": [r for r in rated if not r["kept"]],
              "all": rated}
    results = {}
    for name, rows in groups.items():
        if rows:
            results[name] = {"n": len(rows), **{
                k: round(mean(r[k] for r in rows), 3) for k in RATINGS},
                "all_four": round(mean(all(r[k] for k in RATINGS) for r in rows), 3)}
    print(json.dumps(results, indent=2))
    SCORES_PATH.write_text(json.dumps({
        "note": "Human ratings of generated quiz items (blind to filter result).",
        "rating_definitions": RATINGS,
        "results": results,
    }, indent=2) + "\n", encoding="utf-8")
    print(f"  -> {SCORES_PATH.relative_to(ROOT)}")

In [7]:
if RUN:
    score() if sys.argv[1:] == ["score"] else generate()
else:
    print('RUN is False: showing the saved results below.')

RUN is False: showing the saved results below.


### Results

In [8]:
qg = saved("quiz_generation.json")
s = qg["summary"]
display(pd.Series(s["outcomes"], name="attempts").to_frame())
pd.Series({k: v for k, v in s.items() if not isinstance(v, dict)}, name=qg["settings"]["embedder"])

,attempts
failed round-trip check,46
kept,29
answer too long,3
answer given away by the question,7
no answer,1


chunks                                647.000
usable_chunks                         370.000
topics                                 33.000
attempts                               86.000
well_formed_rate                        0.872
roundtrip_pass_rate_of_well_formed      0.387
kept_rate                               0.337
topics_with_at_least_one_item          22.000
seconds_per_attempt                     1.950
Name: bge-small, dtype: float64

## Calibrating the grader

Calibrates and checks quiz.grade(), the function that marks a student's
free-text answer against a reference answer.

Labelled pairs come from data/eval/generation_analysis.json, which stores 25
generated answers with a correct/incorrect label against the ground-truth
answer:
```text
  - the 25 stored pairs (12 labelled correct, 13 incorrect), with the manual
    review in generation_manual_review.json applied — it changes no label;
  - 25 mismatched pairs: each generated answer scored against the reference of
    a different question, all incorrect by construction.
```

Caveats:
```text
  - the stored labels were produced by a rule (containment, token F1 >= 0.6,
    or all numbers present), so they share token F1 with the grader;
  - every correct pair here is caught by containment or token F1, so the set
    has no true paraphrases and cannot show how high the threshold may go;
  - the number-mismatch rule in quiz.grade() was added after this set showed
    three numeric false positives, so results on it are not an independent
    test of that rule.
```
data/eval/grader_decisions.csv lists every decision for human checking.

In [9]:
# The script's command-line options, as it would have read them.
sys.argv = ['notebook', '--source', 'generation_analysis_sentence_bge-small_hybrid_qwen2.5-1.5b.json']

In [10]:
import csv
import json
import os
import sys
from pathlib import Path

ROOT = REPO_ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from backend.pipeline import quiz  # noqa: E402

EVAL_DIR = ROOT / "data" / "eval"
# The labelled pairs. "--source <name>" scores the grader against a different
# generation_analysis run — the answers a different model wrote, for instance —
# which checks that GRADE_THRESHOLD still separates correct from incorrect
# rather than re-fitting it.
SOURCE_NAME = (sys.argv[sys.argv.index("--source") + 1]
               if "--source" in sys.argv else "generation_analysis.json")
SOURCE = EVAL_DIR / SOURCE_NAME
SUFFIX = ("" if SOURCE_NAME == "generation_analysis.json"
          else "_" + SOURCE_NAME[len("generation_analysis_"):-len(".json")])
OUT_PATH = EVAL_DIR / f"grader_calibration{SUFFIX}.json"
CSV_PATH = EVAL_DIR / f"grader_decisions{SUFFIX}.csv"
THRESHOLDS = [round(0.40 + 0.05 * i, 2) for i in range(12)]   # 0.40 .. 0.95

In [11]:
def build_pairs():
    rows = json.loads(SOURCE.read_text(encoding="utf-8"))["results"]
    pairs = [{"kind": "stored", "question": r["question"],
              "answer": r["generated_answer"], "reference": r["expected_answer"],
              "label": bool(r["answer_correct"])} for r in rows]
    n = len(rows)
    for i, r in enumerate(rows):
        other = rows[(i + 1) % n]
        pairs.append({"kind": "mismatched", "question": other["question"],
                      "answer": r["generated_answer"],
                      "reference": other["expected_answer"], "label": False})
    return pairs

In [12]:
def confusion(pairs, threshold):
    tp = fp = tn = fn = 0
    for p in pairs:
        predicted = p["method"] == "containment" or p["score"] >= threshold
        if predicted and p["label"]:
            tp += 1
        elif predicted:
            fp += 1
        elif p["label"]:
            fn += 1
        else:
            tn += 1
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"threshold": threshold, "tp": tp, "fp": fp, "tn": tn, "fn": fn,
            "accuracy": round((tp + tn) / len(pairs), 3),
            "precision": round(precision, 3), "recall": round(recall, 3),
            "f1": round(f1, 3)}

In [13]:
def main():
    pairs = build_pairs()
    for p in pairs:
        # threshold 2.0 = never pass on score, so .score is the raw evidence
        g = quiz.grade(p["answer"], p["reference"], threshold=2.0)
        p["score"], p["method"] = g.score, g.method

    sweep = [confusion(pairs, t) for t in THRESHOLDS]
    chosen = confusion(pairs, quiz.GRADE_THRESHOLD)
    stored_only = confusion([p for p in pairs if p["kind"] == "stored"],
                            quiz.GRADE_THRESHOLD)

    print(f"{len(pairs)} pairs, {sum(p['label'] for p in pairs)} labelled correct\n")
    print(f"{'thr':>5} {'acc':>6} {'prec':>6} {'rec':>6} {'f1':>6}   tp fp tn fn")
    for s in sweep:
        mark = "  <- GRADE_THRESHOLD" if s["threshold"] == quiz.GRADE_THRESHOLD else ""
        print(f"{s['threshold']:>5} {s['accuracy']:>6} {s['precision']:>6} "
              f"{s['recall']:>6} {s['f1']:>6}   {s['tp']:>2} {s['fp']:>2} "
              f"{s['tn']:>2} {s['fn']:>2}{mark}")
    print("\nstored pairs only:", stored_only)

    disagreements = [p for p in pairs
                     if (p["method"] == "containment"
                         or p["score"] >= quiz.GRADE_THRESHOLD) != p["label"]]
    print(f"\n{len(disagreements)} disagreements at the chosen threshold:")
    for p in disagreements:
        print(f"  [{p['kind']}] label={p['label']} score={p['score']} ({p['method']})"
              f"\n     answer   : {p['answer']}\n     reference: {p['reference']}")

    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "kind", "question", "answer", "reference", "label", "score", "method",
            "grader_correct", "human_correct"])
        writer.writeheader()
        for p in pairs:
            writer.writerow({**p, "grader_correct":
                             p["method"] == "containment" or p["score"] >= quiz.GRADE_THRESHOLD,
                             "human_correct": ""})

    OUT_PATH.write_text(json.dumps({
        "note": "Grader calibration on rule-labelled pairs; see script docstring "
                "for the caveat. Scores use MiniLM cosine and token F1.",
        "pairs": len(pairs),
        "labelled_correct": sum(p["label"] for p in pairs),
        "grade_threshold": quiz.GRADE_THRESHOLD,
        "at_threshold": chosen,
        "at_threshold_stored_pairs_only": stored_only,
        "sweep": sweep,
        "disagreements": disagreements,
    }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"\n  -> {OUT_PATH.relative_to(ROOT)}\n  -> {CSV_PATH.relative_to(ROOT)}")

In [14]:
if RUN:
    main()
else:
    print('RUN is False: showing the saved results below.')

RUN is False: showing the saved results below.


### Results

In [15]:
gc = saved("grader_calibration_sentence_bge-small_hybrid_qwen2.5-1.5b.json")
print(f"{gc['pairs']} labelled pairs, {gc['labelled_correct']} correct; threshold {gc['grade_threshold']}")
display(pd.Series(gc["at_threshold"], name="at the threshold").to_frame().T)
pd.DataFrame(gc["sweep"])

50 labelled pairs, 18 correct; threshold 0.7


,threshold,tp,fp,tn,fn,accuracy,precision,recall,f1
at the threshold,0.7,14.0,1.0,31.0,4.0,0.9,0.933,0.778,0.848


,threshold,tp,fp,tn,fn,accuracy,precision,recall,f1
0,0.40,15,1,31,3,0.92,0.938,0.833,0.882
1,0.45,15,1,31,3,0.92,0.938,0.833,0.882
2,0.50,15,1,31,3,0.92,0.938,0.833,0.882
3,0.55,14,1,31,4,0.90,0.933,0.778,0.848
4,0.60,14,1,31,4,0.90,0.933,0.778,0.848
5,0.65,14,1,31,4,0.90,0.933,0.778,0.848
6,0.70,14,1,31,4,0.90,0.933,0.778,0.848
7,0.75,13,1,31,5,0.88,0.929,0.722,0.813
8,0.80,13,1,31,5,0.88,0.929,0.722,0.813
9,0.85,13,1,31,5,0.88,0.929,0.722,0.813
